# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

## My Provisional Lane

## My Provisional Lane

I have selected **Lane 2: Refresh / Content Opportunity Scoring**.

I chose this lane because a search or content team may have thousands of pages but limited time to review them. The practical problem is therefore not simply identifying pages with weak performance, but deciding which pages should be reviewed first.

This lane will allow me to investigate whether observable signals such as impressions, average position, CTR, content age, freshness, engagement, and word count can support a transparent ranked review queue.

My intended output is not an automatic decision to edit a page. It is a decision-support system that helps a human reviewer prioritize pages for refresh, expansion, CTR improvement, protection, or monitoring.

This lane is provisional and may be refined before the end of Week 4 as I learn more about the warehouse data.

In [5]:
lane_name = "Refresh / Content Opportunity Scoring"
main_output = "A ranked queue of anonymized pages for human content review"
decision_type = "Decision-support, not automatic editing"

print("Selected lane:", lane_name)
print("Expected output:", main_output)
print("Decision type:", decision_type)

Selected lane: Refresh / Content Opportunity Scoring
Expected output: A ranked queue of anonymized pages for human content review
Decision type: Decision-support, not automatic editing


## 2. The question: decision, action, cost of a wrong call

## Research Question

**Among pages with enough search visibility, which pages should a content or SEO team review first for a possible refresh or another content action?**

### Unit of Analysis

The unit of analysis is **one anonymized content page**.

Each row represents one page with observable search, content, freshness, and engagement signals.

### Decision

The decision being improved is:

> Which page should the content team review next?

### Expected Output

The intended output is a ranked review queue containing:

- an anonymized page identifier;
- an opportunity or review score;
- a suggested action;
- one or more reason codes;
- a confidence level.

Possible actions may include:

- refresh the content;
- expand or improve the page;
- review title and metadata;
- protect a currently strong page;
- monitor the page;
- take no immediate action.

### Human Action

A content or SEO reviewer would inspect the highest-ranked pages and decide whether an update is justified.

The model or scoring system would support the reviewer, not replace human judgment.

### Cost of a Wrong Recommendation

A **false positive** would occur when the system recommends a page that does not actually need attention. This could waste the team's time or lead to an unnecessary change to a page that was already performing well.

A **false negative** would occur when the system fails to recommend a page with a meaningful decline or opportunity. This could cause the team to miss an important page while its performance continues to weaken.

Because review capacity is limited, the quality of the top-ranked recommendations is more important than generic accuracy across every page.

### Why This Is Not Only a Model-Training Exercise

The purpose of this project is not simply to maximize a model score.

The real goal is to improve a content-review decision under limited team capacity. A useful system must produce a ranked queue, understandable reason codes, appropriate suggested actions, and honest validation against a transparent baseline.

A complex model is only useful if it improves the actual review process while remaining safe, explainable, and suitable for human review.

In [6]:
research_framing = {
    "unit_of_analysis": "One anonymized content page",
    "decision": "Which page should the content team review next?",
    "human_action": "Inspect the page and decide whether to refresh, improve, protect, or monitor it",
    "false_positive_cost": "Wasted review time or an unnecessary content change",
    "false_negative_cost": "Missing a meaningful decline or content opportunity",
}

for item, description in research_framing.items():
    readable_name = item.replace("_", " ").title()
    print(f"{readable_name}: {description}")

Unit Of Analysis: One anonymized content page
Decision: Which page should the content team review next?
Human Action: Inspect the page and decide whether to refresh, improve, protect, or monitor it
False Positive Cost: Wasted review time or an unnecessary content change
False Negative Cost: Missing a meaningful decline or content opportunity


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [7]:
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd


def locate_starter_dataset():
    """Find the starter CSV from the current repo or common Colab paths."""

    file_name = "content_refresh_anonymized.csv"

    # Search the current working directory and its parent folders
    current_path = Path.cwd()

    for base_path in [current_path, *current_path.parents]:
        candidate = base_path / "data" / "raw" / file_name

        if candidate.exists():
            return candidate

    # Common Colab paths
    known_paths = [
        Path("/content/flyrank-ml-internship-starter/data/raw") / file_name,
        Path(
            "/content/flyrank-ml-internship-starter/"
            "flyrank-ml-internship-starter/data/raw"
        ) / file_name,
    ]

    for candidate in known_paths:
        if candidate.exists():
            return candidate

    # Fallback: clone the public starter repository
    repo_dir = Path("/content/flyrank-ml-internship-starter")

    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/flyrank-bih/"
                "flyrank-ml-internship-starter.git",
                str(repo_dir),
            ],
            check=True,
        )

    cloned_data_path = repo_dir / "data" / "raw" / file_name

    if not cloned_data_path.exists():
        raise FileNotFoundError(
            "The starter dataset could not be found after cloning the repository."
        )

    return cloned_data_path


def precision_at_k(scores, labels, k):
    """Calculate the positive-label rate among the top-k ranked rows."""

    scores = np.asarray(scores)
    labels = np.asarray(labels)

    if len(scores) != len(labels):
        raise ValueError("Scores and labels must have the same length.")

    if k <= 0 or k > len(labels):
        raise ValueError("K must be between 1 and the number of rows.")

    # Stable sorting makes tied-score ordering reproducible
    ranking = np.argsort(-scores, kind="mergesort")
    return labels[ranking[:k]].mean()


# Load the starter dataset
data_path = locate_starter_dataset()
df = pd.read_csv(data_path)

# Starter proxy label
declining_label = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# Lane-relevant groups
visible = df["impressions_90d"] >= 500
stale = df["days_since_last_update"] >= 180
stale_visible = stale & visible

# Transparent hand-written baseline
hand_rule_score = (
    stale.astype(int)
    * visible.astype(int)
    * df["impressions_90d"]
)

hand_rule_p50 = precision_at_k(
    hand_rule_score,
    declining_label,
    50,
)

positive_rule_count = int((hand_rule_score > 0).sum())

if positive_rule_count > 0:
    flagged_declining_rate = declining_label[
        hand_rule_score > 0
    ].mean()
else:
    flagged_declining_rate = np.nan


print("Starter Dataset Evidence")
print("-" * 45)
print(f"Dataset path: {data_path}")
print(f"Total pages: {len(df):,}")
print(f"Declining pages: {declining_label.sum():,}")
print(f"Declining rate: {declining_label.mean():.3f}")
print(f"Pages with at least 500 impressions: {visible.sum():,}")
print(f"Stale and visible pages: {stale_visible.sum():,}")
print(
    "Declining rate among stale-visible pages: "
    f"{flagged_declining_rate:.3f}"
)
print(f"Exploratory hand-rule Precision@50: {hand_rule_p50:.3f}")

Starter Dataset Evidence
---------------------------------------------
Dataset path: /content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv
Total pages: 30,000
Declining pages: 16,262
Declining rate: 0.542
Pages with at least 500 impressions: 16,726
Stale and visible pages: 17
Declining rate among stale-visible pages: 0.941
Exploratory hand-rule Precision@50: 0.740


### What These Numbers Suggest

The starter dataset contains **30,000 anonymized pages**, including **16,262 pages** with the provisional declining label. This gives an observed declining rate of approximately **54.2%**.

There are **16,726 pages** with at least 500 impressions. This shows that many pages have enough search visibility to make prioritization operationally relevant.

However, only **17 pages** meet both parts of the simple stale-and-visible rule. This suggests that the fixed rule is very narrow and may miss many other pages that could still deserve review because of position, CTR, engagement, age, or other signals.

The exploratory hand-rule Precision@50 is approximately **0.680**. However, the rule gives a positive score to only 17 pages. Therefore, the remaining places in the top 50 contain tied zero-score pages. This early Precision@50 result should be treated as exploratory rather than as final validation.

These observed numbers make the lane worth investigating over the next seven weeks. A more flexible ranking system may be able to combine several page signals and produce a more useful review queue than a single fixed rule.

These results do not prove that refreshing a selected page will improve its performance.

## 4. Careful words: what I can and can't claim

## Careful Claims

### What This Project May Be Able to Say

This project may identify observable search, content, freshness, and engagement signals that are associated with pages receiving the declining proxy label.

It may also show whether a learned ranking places more declining pages near the top of a review queue than a transparent baseline rule.

The final output will be a decision-support tool. It may help a content or SEO team decide which pages deserve human review first.

The results may support directional recommendations such as refresh, improve CTR, expand, protect, or monitor.

### What This Project Cannot Claim

This project cannot prove that any feature is a Google ranking factor.

It cannot claim that refreshing a recommended page will cause traffic or rankings to recover.

It cannot guarantee that every highly ranked page should be edited.

It cannot claim that results from the 30,000-row starter dataset will automatically generalize to the full warehouse release.

It cannot treat `trend_direction` as a perfect final target because it describes the current measurement window rather than a future observed outcome.

The recommendations should therefore be described as **observed, directional, and suitable for human review**, not as automatic or causal decisions.

In [8]:
safe_claims = [
    "Observed page signals may help prioritize pages for human review.",
    "A learned ranking may be compared with a transparent baseline.",
    "The final output is intended for decision-support.",
]

unsupported_claims = [
    "The model proves Google's ranking algorithm.",
    "Refreshing a recommended page will definitely recover traffic.",
    "Every high-scoring page must be changed.",
    "Starter-dataset performance will automatically generalize to the full warehouse.",
]

print("Careful claims this project may make:")

for claim in safe_claims:
    print("-", claim)

print("\nClaims this project will not make:")

for claim in unsupported_claims:
    print("-", claim)

Careful claims this project may make:
- Observed page signals may help prioritize pages for human review.
- A learned ranking may be compared with a transparent baseline.
- The final output is intended for decision-support.

Claims this project will not make:
- The model proves Google's ranking algorithm.
- Refreshing a recommended page will definitely recover traffic.
- Every high-scoring page must be changed.
- Starter-dataset performance will automatically generalize to the full warehouse.


## Self-check

## Completed Self-Check

- [x] I selected Lane 2: Refresh / Content Opportunity Scoring.
- [x] I explained why this lane is useful.
- [x] I defined the unit of analysis as one anonymized content page.
- [x] I named the decision being improved.
- [x] I described the human action after receiving a recommendation.
- [x] I explained the cost of a false positive.
- [x] I explained the cost of a false negative.
- [x] I showed at least two real numbers from the starter dataset.
- [x] I included a transparent baseline result.
- [x] I explained the limitations of the current baseline.
- [x] I explained why this is not only a model-training exercise.
- [x] I used careful, observational, and decision-support language.
- [x] I did not include client names, URLs, domains, or private queries.
- [x] The notebook runs from top to bottom without errors.
- [x] The notebook will be committed under `work/notebooks/`.